# S&P500/VIX Report Figures

Report-only notebook for CS673/public figures. It does not train models by default. It compares local paper-style outputs for the public discrete baseline, the best discrete research model, and the continuous reference when those local `outputs/` paths exist.


## Setup


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import pandas as pd
import yaml
from IPython.display import Image, Markdown, display

PUBLIC_BASELINE_OUTPUT_DIR = Path("outputs/sp500_vix_discrete/paper_style")
BEST_DISCRETE_OUTPUT_DIR = Path("outputs/sp500_vix_discrete/best_discrete_research/paper_style")
CONTINUOUS_OUTPUT_DIR = Path("outputs/sp500_vix_continuous/paper_style")
RUN_IF_MISSING = False
AUTO_SELECT_MODEL = True
MODEL_REGISTRY_PATH = "trained_models/model_registry.yaml"
MODEL_SELECTION_PROFILE = "balanced_market"
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_HEAVY = False


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (
            candidate / "configs" / "experiments"
        ).exists():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


REPO_ROOT = find_repo_root()


def repo_path(path: str | Path) -> Path:
    candidate = Path(path).expanduser()
    if candidate.is_absolute():
        return candidate
    return (REPO_ROOT / candidate).resolve()


def display_path(path: str | Path) -> str:
    resolved = repo_path(path)
    try:
        return str(resolved.relative_to(REPO_ROOT))
    except ValueError:
        return str(resolved)


def load_yaml(path: str | Path) -> dict[str, Any]:
    resolved = repo_path(path)
    if not resolved.exists():
        return {}
    loaded = yaml.safe_load(resolved.read_text())
    return loaded if isinstance(loaded, dict) else {}


def load_json(path: str | Path) -> dict[str, Any] | None:
    resolved = repo_path(path)
    if not resolved.exists():
        return None
    return json.loads(resolved.read_text())


OUTPUT_DIRS = [
    {
        "key": "public_baseline",
        "label": "Public discrete baseline",
        "path": PUBLIC_BASELINE_OUTPUT_DIR,
        "comparison_key": "discrete",
        "model": "standard VQ + additive AR",
    },
    {
        "key": "best_discrete_research",
        "label": "Best discrete research model",
        "path": BEST_DISCRETE_OUTPUT_DIR,
        "comparison_key": "discrete",
        "model": "hidden128 VQ + causal conv-transformer k3",
    },
    {
        "key": "continuous_reference",
        "label": "Continuous BetaCVAE reference",
        "path": CONTINUOUS_OUTPUT_DIR,
        "comparison_key": "continuous",
        "model": "continuous BetaCVAE",
    },
]

PAPER_FIGURES = [
    ("real_vs_generated_paths.png", "Real vs generated S&P500/VIX paths."),
    ("returns_distribution.png", "One-step return distribution."),
    ("terminal_return_distribution.png", "Terminal-return distribution."),
    ("volatility_distribution.png", "Path-volatility distribution."),
    ("maximum_drawdown_distribution.png", "Maximum-drawdown distribution."),
    ("squared_return_autocorrelation.png", "Squared-return autocorrelation."),
    ("vix_bucket_paths.png", "Generated paths by VIX bucket."),
    ("vix_bucket_terminal_returns.png", "VIX-bucket terminal-return comparison."),
    ("vix_bucket_volatility.png", "VIX-bucket volatility comparison."),
]

print(f"Repository root: {REPO_ROOT}")
print(
    "Set PUBLIC_BASELINE_OUTPUT_DIR, BEST_DISCRETE_OUTPUT_DIR, and CONTINUOUS_OUTPUT_DIR to local paper-style output directories before running comparisons."
)

## Registered Model Selection


In [ ]:
from time_causal_vae.experiments.model_registry import load_registry, select_registered_model

REPORT_REGISTRY_ROWS = []
if AUTO_SELECT_MODEL:
    registry = load_registry(repo_path(MODEL_REGISTRY_PATH))
    sp500_discrete_candidates = registry["experiments"]["sp500_vix"]["discrete"]["candidates"]
    public_discrete = select_registered_model(
        registry,
        experiment="sp500_vix",
        family="discrete",
        profile=MODEL_SELECTION_PROFILE,
    )
    continuous_reference = select_registered_model(
        registry,
        experiment="sp500_vix",
        family="continuous",
        profile=MODEL_SELECTION_PROFILE,
    )
    research_candidate_id = "conditional_hidden128_conv_transformer_k3"
    research_candidate = sp500_discrete_candidates.get(research_candidate_id)
    OUTPUT_DIRS = [
        {
            "key": "public_baseline",
            "label": "Public discrete baseline",
            "path": PUBLIC_BASELINE_OUTPUT_DIR,
            "comparison_key": "discrete",
            "model": public_discrete.candidate_id,
            "registry_metrics": public_discrete.metrics,
            "missing_metrics": public_discrete.missing_metrics,
            "tokenizer_config": public_discrete.tokenizer_config,
            "prior_config": public_discrete.prior_config,
        },
        {
            "key": "continuous_reference",
            "label": "Continuous reference",
            "path": CONTINUOUS_OUTPUT_DIR,
            "comparison_key": "continuous",
            "model": continuous_reference.candidate_id,
            "registry_metrics": continuous_reference.metrics,
            "missing_metrics": continuous_reference.missing_metrics,
            "config": continuous_reference.config,
        },
    ]
    if research_candidate is not None:
        OUTPUT_DIRS.insert(
            1,
            {
                "key": "best_discrete_research",
                "label": "Best discrete research model",
                "path": BEST_DISCRETE_OUTPUT_DIR,
                "comparison_key": "discrete",
                "model": research_candidate_id,
                "registry_metrics": research_candidate.get("metrics", {}),
                "missing_metrics": research_candidate.get("missing_metrics", []),
                "tokenizer_config": research_candidate.get("tokenizer_config"),
                "prior_config": research_candidate.get("prior_config"),
            },
        )
    for entry in OUTPUT_DIRS:
        row = {
            "role": entry["label"],
            "model": entry["model"],
            "output_dir": display_path(entry["path"]),
        }
        row.update(entry.get("registry_metrics", {}))
        REPORT_REGISTRY_ROWS.append(row)
    display(pd.DataFrame(REPORT_REGISTRY_ROWS))
else:
    print("AUTO_SELECT_MODEL=False; using notebook-local report defaults.")

## Output Manifest


In [ ]:
inventory_rows = []
for entry in OUTPUT_DIRS:
    output_dir = entry["path"]
    summary_path = output_dir / "paper_style_summary.json"
    inventory_rows.append({
        "role": entry["label"],
        "model": entry["model"],
        "output_dir": display_path(output_dir),
        "summary_path": display_path(summary_path),
        "summary_exists": repo_path(summary_path).exists(),
    })
display(pd.DataFrame(inventory_rows))

for row in inventory_rows:
    if row["summary_exists"]:
        continue
    print(
        f"Missing {row['role']} summary at {row['summary_path']}. "
        "Set the matching *_OUTPUT_DIR parameter to a local directory containing "
        "paper_style_summary.json and figure PNGs."
    )

if RUN_IF_MISSING:
    print(
        "RUN_IF_MISSING=True is intentionally non-executing in this report notebook; "
        "run the paper-style evaluation commands outside the notebook after replacing "
        "local checkpoint paths."
    )

## Metric Comparison


In [ ]:
PROFILE_KEYS = ["mmd", "swd", "terminal_return_wasserstein", "volatility_wasserstein"]
METRIC_KEYS = [
    "mmd",
    "swd",
    "terminal_return_wasserstein",
    "volatility_wasserstein",
    "maximum_drawdown_wasserstein",
    "squared_return_autocorrelation_within_path_l1",
]


def select_metrics(
    payload: dict[str, Any], preferred_key: str, label: str
) -> tuple[dict[str, Any], str | None]:
    comparisons = payload.get("comparisons", {})
    if not isinstance(comparisons, dict) or not comparisons:
        print(f"{label}: summary has no comparison metrics.")
        return {}, None
    if preferred_key in comparisons and isinstance(comparisons[preferred_key], dict):
        return comparisons[preferred_key], preferred_key
    fallback_key = next(iter(comparisons))
    print(f"{label}: comparison `{preferred_key}` missing; using `{fallback_key}` instead.")
    fallback = comparisons[fallback_key]
    return fallback if isinstance(fallback, dict) else {}, fallback_key


summary_rows = []
for entry in OUTPUT_DIRS:
    summary_path = entry["path"] / "paper_style_summary.json"
    payload = load_json(summary_path)
    if payload is None:
        continue
    metrics, used_key = select_metrics(payload, entry["comparison_key"], entry["label"])
    if not metrics:
        continue
    profile = None
    if all(key in metrics for key in PROFILE_KEYS):
        profile = sum(float(metrics[key]) for key in PROFILE_KEYS)
    row = {
        "role": entry["label"],
        "model": entry["model"],
        "comparison": used_key,
        "profile": profile,
    }
    row.update({key: metrics.get(key) for key in METRIC_KEYS})
    summary_rows.append(row)

if summary_rows:
    display(pd.DataFrame(summary_rows))
else:
    print(
        "No paper-style summaries were found. Set the output-dir parameters above to "
        "local directories containing paper_style_summary.json."
    )

## Display Existing Figures

Figures are displayed only when the configured local output directories already contain them. Missing files are reported without failing the notebook.


In [ ]:
for entry in OUTPUT_DIRS:
    output_dir = entry["path"]
    if not repo_path(output_dir).exists():
        print(
            f"{entry['label']}: missing output directory {display_path(output_dir)}. "
            "Set the matching *_OUTPUT_DIR parameter to an existing local output path."
        )
        continue
    display(Markdown(f"### {entry['label']}: `{display_path(output_dir)}`"))
    for filename, caption in PAPER_FIGURES:
        path = output_dir / filename
        resolved = repo_path(path)
        if not resolved.exists():
            print(f"missing: {display_path(path)}")
            continue
        display(Markdown(f"#### `{filename}`"))
        display(Markdown(caption))
        display(Image(filename=str(resolved)))

## Missing Output Instructions

The notebook does not train or evaluate by default. If a configured path is missing, generate the corresponding paper-style output outside the notebook, then update the parameters in the setup cell.


In [ ]:
expected_outputs = [
    (entry["label"], entry["path"] / "paper_style_summary.json") for entry in OUTPUT_DIRS
]
for label, summary_path in expected_outputs:
    if repo_path(summary_path).exists():
        print(f"{label}: found {display_path(summary_path)}")
        continue
    print(f"{label}: expected {display_path(summary_path)}")

print(
    "Public baseline output should come from the standard VQ + additive AR paper-style run. "
    "Best discrete output should come from the hidden128 conv-transformer k3 run with "
    "temperature=1.0 and top_k=none. Continuous output should expose the BetaCVAE "
    "reference metrics. The notebook continues when any of these paths are absent."
)
if not RUN_IF_MISSING:
    print("RUN_IF_MISSING=False; no generation commands are executed from this notebook.")

## Interpretation

Standard VQ with the additive AR prior remains the public discrete default because it gives broad code utilisation, VIX-sensitive code usage, and the simplest one-code-per-time-step interface for the additive scalar-conditioned causal AR prior.

Hidden128 VQ with a causal conv-transformer k3 prior is the best discrete research model under the current S&P500/VIX paper-style profile evidence. Report it as a research variant, not as the new public default, and compare it against the continuous BetaCVAE reference.
